In [2]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)


In [15]:
print(sys.path)

['c:\\Python313\\python313.zip', 'c:\\Python313\\DLLs', 'c:\\Python313\\Lib', 'c:\\Python313', '', 'C:\\Users\\Gaayatri Pradeep\\AppData\\Roaming\\Python\\Python313\\site-packages', 'C:\\Users\\Gaayatri Pradeep\\AppData\\Roaming\\Python\\Python313\\site-packages\\win32', 'C:\\Users\\Gaayatri Pradeep\\AppData\\Roaming\\Python\\Python313\\site-packages\\win32\\lib', 'C:\\Users\\Gaayatri Pradeep\\AppData\\Roaming\\Python\\Python313\\site-packages\\Pythonwin', 'c:\\Python313\\Lib\\site-packages', 'c:\\L2-Training\\retail_sales_uc']


In [16]:
from src.utils.ddl_utils import create_tables, drop_tables
from src.models.staging_sql import staging_sql_commands,drop_staging_sql_commands
from src.models.warehouse_sql import warehouse_sql_commands,drop_warehouse_sql_commands


In [17]:
from src.load import load_staging,load_warehouse


In [18]:
from src.utils.connections import get_staging_db, get_target_db

staging_db = get_staging_db()
staging_db.connect()
target_db = get_target_db()
target_db.connect()

In [5]:
drop_tables(staging_db,drop_staging_sql_commands)

Dropping table via: drop_sales
Dropping table via: drop_inventory
Dropping table via: drop_products
Dropping table via: drop_stores
Dropping table via: drop_customers
Dropping table via: drop_supplier
Dropping table via: drop_metadata
Dropping table via: drop dimdate
All tables dropped.


In [6]:
drop_tables(target_db,drop_warehouse_sql_commands)

Dropping table via: drop_fact_sales
Dropping table via: drop_dim_date
Dropping table via: drop_dim_supplier
Dropping table via: drop_dim_store
Dropping table via: drop_dim_product
Dropping table via: drop_dim_customer
All tables dropped.


Creating Tables

In [7]:
create_tables(staging_db,staging_sql_commands)

Creating Table: stg_supplier
stg_supplier created
Creating Table: stg_customers
stg_customers created
Creating Table: stg_stores
stg_stores created
Creating Table: stg_products
stg_products created
Creating Table: stg_inventory
stg_inventory created
Creating Table: stg_sales
stg_sales created
Creating Table: stg_metadata
stg_metadata created
Creating Table: stg_dimdate
stg_dimdate created


In [8]:
create_tables(target_db,warehouse_sql_commands)

Creating Table: dim_customer
dim_customer created
Creating Table: dim_product
dim_product created
Creating Table: dim_store
dim_store created
Creating Table: dim_supplier
dim_supplier created
Creating Table: dim_date
dim_date created
Creating Table: fact_sales
fact_sales created


ETL customers

In [9]:
# 1. Extract and Load CSV to staging area

load_staging.load_csv(staging_db,'../data/customers.csv','customers')



 Starting load for table: customers
No previous load detected. Performing full load...
Extracted Data from ../data/customers.csv
Table 'customers' created and data inserted.
Full load complete. Loaded 10000 rows into customers
Load complete for table: customers


In [10]:
#2 Perform Transformations on staging table

customers = transformation.transform_dim_table(staging_db,'customers',column_mappings.staging_to_dimension_map)


Initial shape: (10000, 6)
Null values per column:
 customer_id      0
customer_name    0
email            0
phone            0
address          0
signup_date      0
dtype: int64
Duplicate rows: 0
Dropped nulls. New shape: (10000, 6)
Dropped duplicates. Final shape: (10000, 6)
Table Transformations applied...


In [11]:
from load.load_warehouse import load_to_data_warehouse

load_to_data_warehouse(target_db,customers,'dim_customer')

Table 'dim_customer' created and data inserted.
Loaded 'dim_customer' to data warehouse with 10000 rows.


ETL for date_dim

In [12]:
load_staging.load_csv(staging_db,'../data/dim_date.csv',"dimdate")


 Starting load for table: dimdate
No previous load detected. Performing full load...
Extracted Data from ../data/dim_date.csv
Table 'dimdate' created and data inserted.
Full load complete. Loaded 2557 rows into dimdate
Load complete for table: dimdate


In [13]:
dates = transformation.transform_dim_table(staging_db,'dimdate',column_mappings.staging_to_dimension_map)

Initial shape: (2557, 6)
Null values per column:
 date           0
day            0
month          0
quarter        0
year           0
day_of_week    0
dtype: int64
Duplicate rows: 0
Dropped nulls. New shape: (2557, 6)
Dropped duplicates. Final shape: (2557, 6)
Table Transformations applied...


In [14]:
load_to_data_warehouse(target_db,dates,'dim_date')

Table 'dim_date' created and data inserted.
Loaded 'dim_date' to data warehouse with 2557 rows.


ETL products

In [15]:
load_staging.load_csv(staging_db,'../data/products.csv','products')


 Starting load for table: products
No previous load detected. Performing full load...
Extracted Data from ../data/products.csv
Table 'products' created and data inserted.
Full load complete. Loaded 500 rows into products
Load complete for table: products


In [16]:
products = transformation.transform_dim_table(staging_db,'products',column_mappings.staging_to_dimension_map)

Initial shape: (500, 4)
Null values per column:
 product_id      0
product_name    0
category        0
price           0
dtype: int64
Duplicate rows: 0
Dropped nulls. New shape: (500, 4)
Dropped duplicates. Final shape: (500, 4)
Table Transformations applied...


In [17]:
load_to_data_warehouse(target_db,products,'dim_product')

Table 'dim_product' created and data inserted.
Loaded 'dim_product' to data warehouse with 500 rows.


ETL supplier

In [18]:
load_staging.load_csv(staging_db,'../data/suppliers.csv','suppliers')


 Starting load for table: suppliers
No previous load detected. Performing full load...
Extracted Data from ../data/suppliers.csv
Table 'suppliers' created and data inserted.
Full load complete. Loaded 200 rows into suppliers
Load complete for table: suppliers


In [19]:
suppliers = transformation.transform_dim_table(staging_db,'suppliers',column_mappings.staging_to_dimension_map)

Initial shape: (800, 4)
Null values per column:
 supplier_id      0
supplier_name    0
contact_name     0
contact_email    0
dtype: int64
Duplicate rows: 600
Dropped nulls. New shape: (800, 4)
Dropped duplicates. Final shape: (200, 4)
Table Transformations applied...


In [20]:
load_to_data_warehouse(target_db,suppliers,'dim_supplier')

Table 'dim_supplier' created and data inserted.
Loaded 'dim_supplier' to data warehouse with 200 rows.


ETL store

In [21]:
load_staging.load_csv(staging_db,'../data/stores.csv','stores')


 Starting load for table: stores
No previous load detected. Performing full load...
Extracted Data from ../data/stores.csv
Table 'stores' created and data inserted.
Full load complete. Loaded 100 rows into stores
Load complete for table: stores


In [22]:
stores = transformation.transform_dim_table(staging_db,'stores',column_mappings.staging_to_dimension_map)

Initial shape: (100, 4)
Null values per column:
 store_id      0
store_name    0
location      0
manager       0
dtype: int64
Duplicate rows: 0
Dropped nulls. New shape: (100, 4)
Dropped duplicates. Final shape: (100, 4)
Table Transformations applied...


In [23]:
load_to_data_warehouse(target_db,stores,'dim_store')

Table 'dim_store' created and data inserted.
Loaded 'dim_store' to data warehouse with 100 rows.


ETL FactSales

In [24]:

load_staging.load_csv(staging_db,'../data/sales.csv','sales')


 Starting load for table: sales
No previous load detected. Performing full load...
Extracted Data from ../data/sales.csv
Table 'sales' created and data inserted.
Full load complete. Loaded 100000 rows into sales
Load complete for table: sales


In [25]:
sales = transformation.transform_fact_sales(staging_db,target_db,column_mappings.staging_to_dimension_map)

Initial shape: (100000, 7)
Null values per column:
 sale_id         0
customer_id     0
product_id      0
store_id        0
sale_date       0
quantity        0
total_amount    0
dtype: int64
Duplicate rows: 0
Dropped nulls. New shape: (100000, 7)
Dropped duplicates. Final shape: (100000, 7)
Fact Sales transformation complete with surrogate keys.


In [26]:
load_to_data_warehouse(target_db,sales,'fact_sales')

Table 'fact_sales' created and data inserted.
Loaded 'fact_sales' to data warehouse with 100000 rows.


In [54]:
from src.utils.validator import DataValidator
from src.extract import csv_loader
from src.utils.validation_schema import schemas

customers = csv_loader.extract_csv('../data/customers.csv')

In [20]:
schemas.keys()

dict_keys(['customers', 'stores', 'sales', 'products', 'inventory', 'supplier'])

In [50]:
schema = schemas.get('sales')

In [33]:
sales_schema = {
    'sale_id': {
        'type': 'string',
        'required': True,
        'minlength': 1
    },
    'customer_id': {
        'type': 'string',
        'required': True,
        'minlength': 1
    },
    'product_id': {
        'type': 'string',
        'required': True,
        'minlength': 1
    },
    'store_id': {
        'type': 'string',
        'required': True,
        'minlength': 1
    },
    'sale_date': {
        'type': 'string',
        'required': True,
        'regex': r'^\d{4}-\d{2}-\d{2}$'  # Matches YYYY-MM-DD
    },
    'quantity': {
        'type': 'integer',
        'required': True,
        'min': 1
    },
    'total_amount': {
        'type': 'float',
        'required': True,
        'min': 0.0
    }
}


In [28]:
records = sales.to_dict(orient = 'records')

In [43]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   sale_id       100000 non-null  object 
 1   customer_id   100000 non-null  object 
 2   product_id    100000 non-null  object 
 3   store_id      100000 non-null  object 
 4   sale_date     100000 non-null  object 
 5   quantity      100000 non-null  int64  
 6   total_amount  100000 non-null  float64
dtypes: float64(1), int64(1), object(5)
memory usage: 5.3+ MB


In [34]:
validator = DataValidator(sales_schema)

In [52]:
schema = schemas.get('customers')

In [55]:
from cerberus import Validator

v = Validator(schema)

def validate_row(row):
    is_valid = v.validate(row.to_dict())
    return is_valid, v.errors.copy() if not is_valid else None

validation_results = customers.apply(validate_row, axis=1)

invalid_rows = [(idx, result[1]) for idx, result in validation_results.items() if not result[0]]

print(f"Found {len(invalid_rows)} invalid rows.")



Found 0 invalid rows.


In [ ]:
def validate_row(row):
    return v.validate(row.to_dict()) , v.errors


In [57]:
import usaddress

def parse_address(addr):
    try:
        parsed, _ = usaddress.tag(addr)
        return {
            'street': parsed.get('AddressNumber', '') + ' ' + parsed.get('StreetName', '') + ' ' + parsed.get('StreetNamePostType', ''),
            'city': parsed.get('PlaceName', ''),
            'state': parsed.get('StateName', ''),
            'zip': parsed.get('ZipCode', '')
        }
    except usaddress.RepeatedLabelError:
        return {'street': '', 'city': '', 'state': '', 'zip': ''}


3. Calculate Customer Retention

In [60]:
parse_address('South Christinaview, OR 05031, 342 Bray Hill')

{'street': '342 Bray Hill ',
 'city': 'South Christinaview',
 'state': 'OR',
 'zip': '05031'}